# Chapter 2: Editing and Filtering Data in Pandas

A practical workbook covering data cleaning, conditional subsetting, and feature engineering:
1. **2.1 Handling Missing Data**: Identifying blanks (`.isna()`), removing rows/columns (`.dropna()`), and filling empty values (`.fillna()`)
2. **2.2 Boolean Filtering & Conditions**: Subsetting data using comparisons, bitwise operators (`&`, `|`, `~`), and the parentheses rule
3. **2.3 Data Transformations**: Creating calculated columns, `.apply()`, mapping categorical values (`.map()`), renaming columns (`.rename()`), and type conversion (`.astype()`)

---
---

## 2.1 Handling Missing Data

### Concept Summary
Real-world data is rarely complete. Sensors go offline, survey participants skip questions, and database exports produce empty slots. Pandas recognizes two primary missing-value representations:
- **`np.nan` (Not a Number):** Used for numeric missing values.
- **`None`:** Standard Python object used for missing text or non-numeric types.

---

### Essential Tools for Missing Data

#### 1. Detecting Missing Values (`df.isna()` or `df.isnull()`)
Returns a DataFrame of the exact same dimensions filled with booleans: `True` if a cell is missing, `False` if valid.
- **Example:**
  ```python
  df = pd.DataFrame({'Name': ['Alice', None], 'Age': [25, np.nan]})
  print(df.isna())
  # Output:
  #     Name    Age
  # 0  False  False
  # 1   True   True
  ```
- **Quick Count of Nulls per Column:**
  ```python
  df.isna().sum()  # Counts total True (missing) values per column
  # Output:
  # Name    1
  # Age     1
  ```

#### 2. Dropping Missing Data (`df.dropna()`)
Removes rows or columns that contain missing values.
- **`axis=0` (Default):** Drops entire **rows** that contain missing entries.
- **`axis=1`:** Drops entire **columns** that contain missing entries.
- **`how='any'` (Default):** Drops the row/column if **even one** single value is missing.
- **`how='all'`:** Drops the row/column **only if every single cell** in it is missing.
- **Example:**
  ```python
  # Drop any row containing at least one NaN
  clean_rows = df.dropna(axis=0, how='any')

  # Drop only columns where ALL values are NaN
  clean_cols = df.dropna(axis=1, how='all')
  ```

#### 3. Imputing / Filling Missing Values (`df.fillna()`) 🧠
Instead of deleting rows, you can substitute missing slots with meaningful replacements:
- **Fixed Value:** Replace with a constant number or string.
  ```python
  df.fillna(0)             # Replaces all NaNs with 0
  df.fillna({'Name': 'Unknown', 'Age': 0})  # Replaces missing Name with 'Unknown' and Age with 0
  df['Age'].fillna(df['Age'].mean())  # Replaces missing age with the average age
  ```
- **Forward Fill (`method='ffill'`):** Carries the previous valid value downward into the missing cell.
  ```python
  df['Score'].ffill()  # Forward fills missing scores from the last valid score
  ```

- **Backward Fill (`method='bfill'`):** Pulls the next valid value upward into the missing cell.

In [3]:
# --- Demo: Detecting, Dropping, and Imputing Missing Values ---
import pandas as pd
import numpy as np


# Sample DataFrame with missing numbers and text
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', None, 'David'],
    'Age': [25.0, np.nan, 30.0, 22.0],
    'Score': [88.0, 92.0, np.nan, np.nan]
})

print("=== Original Table ===")
print(df)



# 1. Boolean detection
print("\n\n----- Demo Question 1 -----")
print("\n=== Missing Values Map (isna) ===")

print(df.isna())
print("\nTotal Missing per Column:\n", df.isna().sum())



# 2. Dropping rows with any missing values
print("\n\n----- Demo Question 2 -----")

print("\n=== Rows Without Any NaNs (dropna) ===")
print(df.dropna(axis=0, how='any'))



# 3. Filling missing entries
print("\n\n----- Demo Question 3 -----")
print("\n=== Filled with Constants ===")

print(df.fillna({'Name': 'Unknown', 'Age': 0, 'Score': 50}))



# 4. Forward filling
print("\n\n----- Demo Question 4 -----")
print("\n=== Forward Filled Scores ===")

print(df['Score'].ffill())

=== Original Table ===
    Name   Age  Score
0  Alice  25.0   88.0
1    Bob   NaN   92.0
2   None  30.0    NaN
3  David  22.0    NaN


----- Demo Question 1 -----

=== Missing Values Map (isna) ===
    Name    Age  Score
0  False  False  False
1  False   True  False
2   True  False   True
3  False  False   True

Total Missing per Column:
 Name     1
Age      1
Score    2
dtype: int64


----- Demo Question 2 -----

=== Rows Without Any NaNs (dropna) ===
    Name   Age  Score
0  Alice  25.0   88.0


----- Demo Question 3 -----

=== Filled with Constants ===
      Name   Age  Score
0    Alice  25.0   88.0
1      Bob   0.0   92.0
2  Unknown  30.0   50.0
3    David  22.0   50.0


----- Demo Question 4 -----

=== Forward Filled Scores ===
0    88.0
1    92.0
2    92.0
3    92.0
Name: Score, dtype: float64


<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 1: Cleaning and Imputing Incomplete Records

**Goal:** Detect missing entries in a patient intake table, evaluate row dropping strategies, and fill empty slots using specific replacement strategies.

**Requirements:**
1. Create a DataFrame named `patients` with the following columns:
   - `'Patient'`: `['John', 'Sara', 'Mike', None, 'Emma']`
   - `'HeartRate'`: `[72, np.nan, 80, 68, np.nan]`
   - `'Cholesterol'`: `[195, 210, np.nan, 180, 240]`
   - `'Notes'`: `[None, None, None, None, None]`
2. Display the count of missing entries for each column.
3. Drop the `'Notes'` column entirely using `.dropna()` with the appropriate axis and condition parameter (`how='all'`).
4. Create a copy of the remaining table and remove every row that still contains any missing value.
5. Starting from the table in step 3, fill missing `'HeartRate'` entries with the average heart rate, and replace any missing `'Patient'` name with the text `'Anonymous'`.

<details>
<summary>💡 Need a hint?</summary>

- Which function counts null values when paired with `.sum()`?
- For step 3, review how `axis=1` selects columns and `how='all'` prevents deleting columns with valid data.
- For step 4, remember that default `dropna()` targets rows with `axis=0`.
- For step 5, pass a dictionary mapping column names to desired fill values into `.fillna()`.
</details>

In [4]:
import pandas as pd


df = pd.DataFrame({
    'Patient': ['John', 'Sara', 'Mike', None, 'Emma'],
    'HeartRate': [72, np.nan, 80, 68, np.nan],
    'Cholesterol': [195, 210, np.nan, 180, 240],
    'Notes': [None, None, None, None, None]
    })

df


,Patient,HeartRate,Cholesterol,Notes
0,John,72.0,195.0,None
1,Sara,NaN,210.0,None
2,Mike,80.0,NaN,None
3,None,68.0,180.0,None
4,Emma,NaN,240.0,None


2. Display the count of missing entries for each column.

In [7]:
df.isna().sum()

Patient        1
HeartRate      2
Cholesterol    1
Notes          5
dtype: int64

3. Drop the `'Notes'` column entirely using `.dropna()` with the appropriate axis and condition parameter (`how='all'`)

In [9]:
df.dropna(axis=1, how='all')

,Patient,HeartRate,Cholesterol
0,John,72.0,195.0
1,Sara,NaN,210.0
2,Mike,80.0,NaN
3,None,68.0,180.0
4,Emma,NaN,240.0


4. Create a copy of the remaining table and remove every row that still contains any missing value.

In [13]:
df_new = df.dropna(axis=1, how='all')

# Copy of remaining table
print(df_new)

df_new.dropna(axis=1, how='any')

  Patient  HeartRate  Cholesterol
0    John       72.0        195.0
1    Sara        NaN        210.0
2    Mike       80.0          NaN
3    None       68.0        180.0
4    Emma        NaN        240.0


""
0
1
2
3
4


5. Starting from the table in step 3, fill missing `'HeartRate'` entries with the average heart rate, and replace any missing `'Patient'` name with the text `'Anonymous'`.

In [22]:
# Let's see heart rate mean
print(df['HeartRate'].mean())
print(f"{df['HeartRate'].mean():.2f}")

df_new.fillna({
    'HeartRate':f"{df['HeartRate'].mean():.2f}",
    'Patient': 'Anonymous'
    })

73.33333333333333
73.33


,Patient,HeartRate,Cholesterol
0,John,72.0,195.0
1,Sara,73.33,210.0
2,Mike,80.0,NaN
3,Anonymous,68.0,180.0
4,Emma,73.33,240.0


<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 2.2 Filtering Data with Boolean Conditions

### Concept Summary
Filtering extracts rows that satisfy specific logical rules (e.g., finding all transactions above $100 or customers from a particular city).

---

### Creating and Applying Boolean Masks

#### 1. Single Condition Filter
Evaluating a condition on a column produces a Series of `True`/`False` flags matching the table's row count.
```python
df = pd.DataFrame({
    'Category': ['A', 'B', 'A', 'B'],
    'Value': [10, 25, 30, 5]
})

mask = df['Value'] > 15
# Applying the mask extracts rows matching True:
filtered_df = df[mask]
```

#### 2. Combining Multiple Conditions (The Bitwise Rules)
Standard Python keywords like `and` or `or` evaluate a single truth value and will fail on Pandas Series. You must use **bitwise logical operators**:

| Operator | Purpose | Behavior |
| :---: | :---: | :--- |
| `&` | **AND** | Row is kept only if **both** conditions are `True` |
| `\|` | **OR** | Row is kept if **either** condition is `True` |
| `~` | **NOT** | Inverts the filter (keeps rows that are `False`) |

> **The Parentheses Rule:** You must wrap **every individual condition in parentheses `(...)`**! Bitwise operators have higher evaluation priority than comparison operators (`>`, `==`), so missing parentheses will cause syntax errors.

- **Combined Example:**
  ```python
  # Filter category 'A' with value over 15:
  mask_and = (df['Category'] == 'A') & (df['Value'] > 15)
  print(df[mask_and])

  # Filter category 'A' OR value under 10:
  mask_or = (df['Category'] == 'A') | (df['Value'] < 10)
  print(df[mask_or])
  ```

In [ ]:
# --- Demo: Multi-Condition Boolean Filtering ---
import pandas as pd

sales_data = pd.DataFrame({
    'Department': ['Electronics', 'Clothing', 'Electronics', 'Home', 'Clothing'],
    'UnitsSold': [12, 45, 8, 30, 15],
    'Revenue': [1200, 900, 800, 1500, 300]
})

print("=== Full Sales Table ===")
print(sales_data)

# 1. Single condition: Revenue >= 1000
print("\n=== Revenue >= 1000 ===")
print(sales_data[sales_data['Revenue'] >= 1000])

# 2. Bitwise AND (&): Electronics with Revenue > 1000
mask_high_elec = (sales_data['Department'] == 'Electronics') & (sales_data['Revenue'] > 1000)
print("\n=== High Revenue Electronics ===")
print(sales_data[mask_high_elec])

# 3. Bitwise OR (|) with NOT (~): Home items OR UnitsSold > 20
mask_combo = (sales_data['Department'] == 'Home') | (sales_data['UnitsSold'] > 20)
print("\n=== Home or Units > 20 ===")
print(sales_data[mask_combo])

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 2: Multi-Condition Filtering on Real Records

**Goal:** Apply compound logical criteria to isolate specific operational records from a warehouse inventory table.

**Requirements:**
1. Create a DataFrame named `warehouse` with the following columns:
   - `'Item'`: `['Monitor', 'Desk', 'Chair', 'Mouse', 'Keyboard', 'Lamp']`
   - `'Category'`: `['Office', 'Furniture', 'Furniture', 'Office', 'Office', 'Furniture']`
   - `'Stock'`: `[25, 8, 12, 120, 85, 4]`
   - `'Price'`: `[180, 250, 95, 20, 45, 30]`
2. Filter and display all items where `'Category'` is `'Office'` and `'Stock'` is strictly greater than `50`.
3. Filter and display all items where `'Price'` is strictly greater than `100` OR `'Stock'` is less than `10`.
4. Using the negation operator (`~`), display all records where `'Category'` is **not** `'Furniture'`.

<details>
<summary>💡 Need a hint?</summary>

- Remember to place each individual condition inside its own parentheses: `(condition_a) & (condition_b)`.
- For step 3, use the pipe operator `|` to represent logical OR.
- For step 4, wrap the entire category equality statement in `~(...)` to invert it.
</details>

In [ ]:
# --- Exercise 2: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 2.3 Data Transformations & Feature Engineering

### Concept Summary
Data transformation means reshaping existing columns, calculating new features, or changing data types to prepare a dataset for analysis or machine learning.

---

### Core Transformation Methods

#### 1. Arithmetic Column Creation
You can compute new columns instantly using vectorized math without writing loops:
```python
df['Total'] = df['Price'] * df['Quantity']
```

#### 2. Transforming via Functions (`df.apply()`)
Applies a custom function or inline `lambda` across elements:
- **Across a Single Column (Series):**
  ```python
  # Squares every number in column 'A'
  df['Square_A'] = df['A'].apply(lambda x: x ** 2)
  ```
- **Across Multiple Columns Row-by-Row (`axis=1`):**
  When you need logic that reads multiple columns at once per row, set `axis=1`:
  ```python
  # Multiplies column A by column B for every row
  df['Product'] = df.apply(lambda row: row['A'] * row['B'], axis=1)
  ```

#### 3. Value Substitution (`Series.map()`)
Replaces specific entries in a column using a lookup dictionary. Very helpful for encoding categories or translating status codes:
```python
# Maps integer levels to word descriptions
df['Level_Name'] = df['Level'].map({1: 'Junior', 2: 'Mid', 3: 'Senior'})
```

#### 4. Renaming Columns (`df.rename()`)
Changes headers using a dictionary mapping `{old_name: new_name}`:
```python
df = df.rename(columns={'A': 'Alpha', 'B': 'Beta'})
```

#### 5. Type Conversion (`Series.astype()`)
Casts column data into a different technical type (`int`, `float`, `str`, `bool`):
```python
df['ID'] = df['ID'].astype(str)          # Converts integers to text
df['Price'] = df['Price'].astype(float)  # Ensures decimal values
```

In [ ]:
# --- Demo: Column Math, apply, map, rename, and astype ---
import pandas as pd

df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6]
})

print("=== Starting Table ===")
print(df)

# 1. Arithmetic addition
df['C'] = df['A'] + df['B']

# 2. apply() with axis=1 (row-wise product)
df['D'] = df.apply(lambda row: row['A'] * row['B'], axis=1)

# 3. map() with a dictionary
df['E'] = df['A'].map({1: 'One', 2: 'Two', 3: 'Three'})

# 4. apply() on a single column (square)
df['F'] = df['A'].apply(lambda x: x ** 2)

# 5. rename() headers
df = df.rename(columns={'A': 'Alpha', 'B': 'Beta'})

# 6. astype() type conversion
df['Alpha'] = df['Alpha'].astype(float)

print("\n=== Transformed Table ===")
print(df)
print("\nData Types:\n", df.dtypes)

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 3: Column Transformations & Type Casting

**Goal:** Apply arithmetic computations, function mappings, categorical label decoding, and data type updates to build an e-commerce order summary.

**Requirements:**
1. Create a DataFrame named `orders` with the following columns:
   - `'OrderID'`: `[1001, 1002, 1003, 1004]`
   - `'UnitPrice'`: `[15.0, 30.0, 8.5, 50.0]`
   - `'Quantity'`: `[2, 1, 4, 3]`
   - `'StatusCode'`: `[1, 2, 1, 3]`
2. Add a calculated column `'Subtotal'` by multiplying `'UnitPrice'` by `'Quantity'`.
3. Using `.apply()` with a `lambda` function on `'Subtotal'`, create a column `'Tax'` that computes 8% tax (`Subtotal * 0.08`).
4. Using `.map()`, translate the `'StatusCode'` integers into human-readable labels in a new column `'StatusDesc'`:
   - `1` $\rightarrow$ `'Processing'`
   - `2` $\rightarrow$ `'Shipped'`
   - `3` $\rightarrow$ `'Delivered'`
5. Rename `'OrderID'` to `'Order_Number'`.
6. Convert the `'Order_Number'` column from integer to string (`str`) using `.astype()` and verify your table's `.dtypes`.

<details>
<summary>💡 Need a hint?</summary>

- For step 2, standard column multiplication works element-wise: `df['Col1'] * df['Col2']`.
- For step 3, write `lambda x: x * 0.08` inside `.apply()` on the Subtotal column.
- For step 4, pass a dictionary mapping numbers to text strings into `df['StatusCode'].map(...)`.
- For step 5, pass `columns={'OldName': 'NewName'}` into `.rename()`.
- For step 6, remember `df['Col'] = df['Col'].astype(str)`.
</details>

In [ ]:
# --- Exercise 3: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>